# 18b — Text-to-SQL recommendation pattern (airline cabin upgrades)

A walkthrough of the **text-to-SQL** pattern from the AWS GenAI Pro exam
scenario: a SaaS app recommends cabin upgrades by querying customer
travel history that lives in a **relational database**, not a vector
index.

> *Why text-to-SQL and not RAG?* The data is **structured and
> authoritative** — booking counts, fare classes, loyalty tier, last
> flight date. Embedding-similarity over RDS rows would invent fuzzy
> matches where the business needs exact joins and aggregates. RAG is
> the wrong shape; text-to-SQL is.

### Pipeline

```
Customer request
   │
   ▼
Bedrock model interprets intent → which tables/columns are needed
   │
   ▼
Text-to-SQL generation (Nova Pro) → candidate SQL
   │
   ▼
SQL validation (allow-list parse, SELECT-only, no DDL/DML, table allow-list)
   │
   ▼
RDS Data API executes the validated SQL → exact rows
   │
   ▼
Bedrock model generates a cabin-upgrade recommendation grounded in the rows
   │
   ▼
Bedrock Guardrails filter the output
   │
   ▼
Step Functions + Lambda orchestrate retries, validation gates, audit logging
   │
   ▼
Application sends the personalized offer
```

This notebook implements every step except the Step Functions deploy —
that part is described in a final cell because it's infrastructure, not
a notebook demo.

Aurora Serverless v2 (PostgreSQL) stands in for the RDS database. The
RDS Data API is the same shape on either, and the OrgIQ-Aurora stack is
already deployed.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import re
import time
from pathlib import Path

config = json.loads(Path("config.json").read_text())
REGION             = config["REGION"]
AURORA_CLUSTER_ARN = config["AURORA_CLUSTER_ARN"]
AURORA_SECRET_ARN  = config["AURORA_SECRET_ARN"]
DB_NAME            = config.get("AURORA_DB_NAME", "orgiq")
MODEL_ID           = config.get("MODEL_ID", "amazon.nova-pro-v1:0")
GUARDRAIL_ID       = config.get("GUARDRAIL_ID")          # optional — set after notebook 05
GUARDRAIL_VERSION  = config.get("GUARDRAIL_VERSION", "DRAFT")

rds = boto3.client("rds-data", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

def exec_sql(sql: str, params=None):
    """Execute via RDS Data API. Returns the raw response."""
    kwargs = dict(
        resourceArn=AURORA_CLUSTER_ARN,
        secretArn=AURORA_SECRET_ARN,
        database=DB_NAME,
        sql=sql,
        includeResultMetadata=True,
    )
    if params:
        kwargs["parameters"] = params
    return rds.execute_statement(**kwargs)

print(f"✓ Config loaded (region={REGION}, model={MODEL_ID})")
print(f"  Aurora: {AURORA_CLUSTER_ARN.split(':cluster:')[-1]}")
print(f"  Guardrail: {GUARDRAIL_ID or '(not set — notebook will skip the guardrail step)'}")

## Step 2: Seed a small airline booking schema in Aurora

Three tables, demo-scale, dropped at the end of the notebook. Schema
mirrors what an airline SaaS would have in RDS: customers, their
bookings, and a loyalty enrichment view.

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS demo_airline_customers (
  customer_id    TEXT PRIMARY KEY,
  full_name      TEXT NOT NULL,
  loyalty_tier   TEXT NOT NULL CHECK (loyalty_tier IN ('basic','silver','gold','platinum')),
  miles_balance  INTEGER NOT NULL DEFAULT 0,
  signup_date    DATE NOT NULL
);

CREATE TABLE IF NOT EXISTS demo_airline_bookings (
  booking_id     TEXT PRIMARY KEY,
  customer_id    TEXT NOT NULL REFERENCES demo_airline_customers(customer_id),
  flight_date    DATE NOT NULL,
  origin         TEXT NOT NULL,
  destination    TEXT NOT NULL,
  cabin_class    TEXT NOT NULL CHECK (cabin_class IN ('economy','premium_economy','business','first')),
  fare_paid_usd  NUMERIC(10,2) NOT NULL,
  upgrade_taken  BOOLEAN NOT NULL DEFAULT FALSE
);

CREATE INDEX IF NOT EXISTS demo_airline_bookings_customer_idx
  ON demo_airline_bookings(customer_id, flight_date DESC);
"""

for stmt in [s for s in SCHEMA.split(";") if s.strip()]:
    exec_sql(stmt + ";")

print("✓ demo_airline_customers + demo_airline_bookings created")

In [ ]:
# Seed a tiny synthetic dataset (4 customers, ~16 bookings) so the model has
# something concrete to recommend over.
SEED_CUSTOMERS = [
    ("c-1001", "Ada Patel",      "platinum", 184_500, "2018-04-11"),
    ("c-1002", "Bram Olsen",     "gold",      78_200, "2020-09-30"),
    ("c-1003", "Chiamaka Eze",   "silver",    24_100, "2022-02-14"),
    ("c-1004", "Diego Marquez",  "basic",      3_800, "2025-11-02"),
]

SEED_BOOKINGS = [
    # Ada — frequent long-haul, business class, never declines an upgrade
    ("b-001", "c-1001", "2026-01-12", "JFK", "LHR", "business",       4200.00, True),
    ("b-002", "c-1001", "2026-02-20", "JFK", "NRT", "business",       5100.00, True),
    ("b-003", "c-1001", "2026-03-15", "LAX", "SYD", "first",          8400.00, False),
    ("b-004", "c-1001", "2026-04-22", "JFK", "CDG", "business",       4350.00, True),
    # Bram — premium economy regular, took upgrade once
    ("b-005", "c-1002", "2026-01-05", "ORD", "FRA", "premium_economy", 1800.00, False),
    ("b-006", "c-1002", "2026-02-18", "ORD", "AMS", "premium_economy", 1900.00, True),
    ("b-007", "c-1002", "2026-03-30", "ORD", "DUB", "economy",          950.00, False),
    ("b-008", "c-1002", "2026-04-25", "ORD", "LIS", "premium_economy", 1750.00, False),
    # Chiamaka — economy, growing loyalty miles, occasional upgrade
    ("b-009", "c-1003", "2026-01-22", "LOS", "DXB", "economy",          780.00, True),
    ("b-010", "c-1003", "2026-02-15", "LOS", "JNB", "economy",          640.00, False),
    ("b-011", "c-1003", "2026-04-04", "LOS", "LHR", "premium_economy", 1620.00, False),
    # Diego — brand new, only economy
    ("b-012", "c-1004", "2026-03-10", "MEX", "MIA", "economy",          420.00, False),
    ("b-013", "c-1004", "2026-04-19", "MEX", "JFK", "economy",          560.00, False),
]

for cid, name, tier, miles, signup in SEED_CUSTOMERS:
    exec_sql(
        "INSERT INTO demo_airline_customers VALUES (:cid, :name, :tier, :miles, CAST(:signup AS DATE)) ON CONFLICT DO NOTHING",
        [
            {"name": "cid",    "value": {"stringValue": cid}},
            {"name": "name",   "value": {"stringValue": name}},
            {"name": "tier",   "value": {"stringValue": tier}},
            {"name": "miles",  "value": {"longValue":   miles}},
            {"name": "signup", "value": {"stringValue": signup}},
        ],
    )

for bid, cid, fdate, orig, dest, cabin, fare, upg in SEED_BOOKINGS:
    exec_sql(
        """INSERT INTO demo_airline_bookings VALUES
           (:bid, :cid, CAST(:fdate AS DATE), :orig, :dest, :cabin, :fare, :upg)
           ON CONFLICT DO NOTHING""",
        [
            {"name": "bid",   "value": {"stringValue": bid}},
            {"name": "cid",   "value": {"stringValue": cid}},
            {"name": "fdate", "value": {"stringValue": fdate}},
            {"name": "orig",  "value": {"stringValue": orig}},
            {"name": "dest",  "value": {"stringValue": dest}},
            {"name": "cabin", "value": {"stringValue": cabin}},
            {"name": "fare",  "value": {"doubleValue": fare}},
            {"name": "upg",   "value": {"booleanValue": upg}},
        ],
    )

count = exec_sql("SELECT COUNT(*) FROM demo_airline_bookings;")
print(f"✓ Seeded {count['records'][0][0]['longValue']} bookings across {len(SEED_CUSTOMERS)} customers")

## Step 3: Interpret the customer request

The first model call doesn't write SQL — it just decides **which tables
and columns are relevant**. Splitting "what do I need?" from "write the
SQL" is what makes the pipeline auditable: each step has a single job
and a small failure surface.

In [ ]:
CUSTOMER_REQUEST = (
    "Customer c-1001 is booking a flight from JFK to LHR next month. "
    "Should we offer them a cabin upgrade, and which cabin?"
)

SCHEMA_DESCRIPTION = """
TABLE demo_airline_customers (customer_id, full_name, loyalty_tier, miles_balance, signup_date)
TABLE demo_airline_bookings  (booking_id, customer_id, flight_date, origin, destination,
                              cabin_class, fare_paid_usd, upgrade_taken)
"""

INTENT_PROMPT = f"""You are an analyst at an airline SaaS. A downstream model will write SQL
against the schema below to answer the customer request. Your job is to decide which
tables and columns the SQL will need.

Schema:
{SCHEMA_DESCRIPTION}

Customer request:
{CUSTOMER_REQUEST}

Reply with ONLY a JSON object:
{{
  "tables":  ["demo_airline_customers", "demo_airline_bookings"],
  "columns": ["..."],
  "reasoning": "one sentence"
}}
"""

resp = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": INTENT_PROMPT}]}],
    inferenceConfig={"maxTokens": 400, "temperature": 0.0},
)
intent_raw = resp["output"]["message"]["content"][0]["text"].strip()
# Strip code fences if the model added them
intent_raw = re.sub(r"^```(?:json)?|```$", "", intent_raw, flags=re.MULTILINE).strip()
intent = json.loads(intent_raw)

print("✓ Intent parsed")
print(json.dumps(intent, indent=2))

## Step 4: Generate SQL (text-to-SQL)

Hand the intent plus the schema to Nova Pro and ask for SQL. Important
prompt-engineering bits:

- **Constrain the dialect** (`PostgreSQL`) so the model doesn't emit MySQL
  backticks or SQL Server `TOP`.
- **Tell it the customer ID is a parameter**, not a literal. Concatenating
  user input into SQL is how you get SQL injection — even when the user
  input is being interpreted by an LLM.
- **Ask for SELECT only**. The validator below double-checks this, but
  setting the expectation up front shrinks the failure surface.

In [ ]:
SQL_PROMPT = f"""You are writing a PostgreSQL query for an airline cabin-upgrade
recommendation system.

Schema (only these tables/columns may appear in the query):
{SCHEMA_DESCRIPTION}

Customer request:
{CUSTOMER_REQUEST}

Rules:
- Use a single SELECT statement (no INSERT/UPDATE/DELETE/DDL).
- PostgreSQL dialect.
- Use the named parameter `:customer_id` instead of inlining 'c-1001'.
- Aggregate booking history (count of trips, count of upgrades taken,
  average fare paid, most common cabin class) plus the customer's
  loyalty_tier and miles_balance.

Return ONLY the SQL — no commentary, no code fences.
"""

resp = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": SQL_PROMPT}]}],
    inferenceConfig={"maxTokens": 600, "temperature": 0.0},
)
candidate_sql = resp["output"]["message"]["content"][0]["text"].strip()
candidate_sql = re.sub(r"^```(?:sql)?|```$", "", candidate_sql, flags=re.MULTILINE).strip()

print("Generated SQL:\n")
print(candidate_sql)

## Step 5: Validate the SQL before execution

Never run LLM-authored SQL straight against the database. Three checks,
in order of cost:

1. **Statement count** — exactly one statement (no stacked queries).
2. **Allow-listed verb** — must start with `SELECT` or `WITH`. No DDL,
   no DML.
3. **Allow-listed tables** — every identifier that looks like a table
   reference must be in the schema we exposed to the model. This catches
   hallucinated joins to `users` or `pg_catalog.*`.

This is a deliberately small validator — production deployments
typically wrap it with `sqlglot` for full AST parsing and column-level
allow-listing, plus a read-only DB role as a second line of defense.

In [ ]:
ALLOWED_TABLES = {"demo_airline_customers", "demo_airline_bookings"}
FORBIDDEN_KEYWORDS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|TRUNCATE|GRANT|REVOKE|COPY|MERGE|VACUUM)\b",
    re.IGNORECASE,
)


class SqlValidationError(Exception):
    pass


def validate_sql(sql: str) -> str:
    sql_stripped = sql.strip().rstrip(";").strip()

    # 1. Single statement
    if ";" in sql_stripped:
        raise SqlValidationError("multiple statements detected — only one SELECT permitted")

    # 2. SELECT/WITH only
    head = sql_stripped.lstrip("(").split(None, 1)[0].upper()
    if head not in ("SELECT", "WITH"):
        raise SqlValidationError(f"statement must start with SELECT or WITH, got {head!r}")
    if FORBIDDEN_KEYWORDS.search(sql_stripped):
        raise SqlValidationError("statement contains a forbidden DDL/DML keyword")

    # 3. Allow-listed tables
    referenced = set(re.findall(r"\bdemo_airline_\w+", sql_stripped, re.IGNORECASE))
    referenced_lower = {t.lower() for t in referenced}
    if not referenced_lower:
        raise SqlValidationError("no allow-listed tables referenced")
    illegal = referenced_lower - ALLOWED_TABLES
    if illegal:
        raise SqlValidationError(f"references disallowed tables: {sorted(illegal)}")

    return sql_stripped


try:
    safe_sql = validate_sql(candidate_sql)
    print("✓ SQL passed validation")
    print(f"\nValidated SQL (will execute next):\n{safe_sql}")
except SqlValidationError as e:
    print(f"✗ Validation failed: {e}")
    print("In production this would trigger a retry through Step Functions")
    print("(regenerate with stricter prompt, or fall back to a templated query).")
    raise

## Step 6: Execute against RDS Data API

In [ ]:
customer_id = "c-1001"
result = exec_sql(
    safe_sql,
    [{"name": "customer_id", "value": {"stringValue": customer_id}}],
)

# Flatten the Data API column metadata + records into plain dicts
columns = [m["name"] for m in result["columnMetadata"]]


def cell_value(c):
    # Data API wraps every value in a typed dict — unwrap whichever key is present
    if "isNull" in c and c["isNull"]:
        return None
    for k in ("stringValue", "longValue", "doubleValue", "booleanValue"):
        if k in c:
            return c[k]
    return next(iter(c.values()))


rows = [{col: cell_value(c) for col, c in zip(columns, rec)} for rec in result["records"]]

print(f"✓ Query returned {len(rows)} row(s)")
print(json.dumps(rows, indent=2, default=str))

## Step 7: Generate the recommendation grounded in the rows

The model now sees only the validated rows — not the raw customer
request as a SQL prompt. That separation is what makes the answer
*grounded in exact data* rather than a confident-sounding guess.

In [ ]:
RECOMMENDATION_PROMPT = f"""You are recommending a cabin upgrade for an airline customer.
Use ONLY the booking-history rows below; do not infer facts that aren't supported by them.

Customer request:
{CUSTOMER_REQUEST}

Booking history (from validated SQL):
{json.dumps(rows, indent=2, default=str)}

Reply with:
1. The recommended cabin (economy / premium_economy / business / first), or "no upgrade".
2. One short sentence of justification, citing specific numbers from the rows.
3. A confidence rating (low / medium / high).
"""

resp = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": RECOMMENDATION_PROMPT}]}],
    inferenceConfig={"maxTokens": 400, "temperature": 0.2},
)
recommendation = resp["output"]["message"]["content"][0]["text"].strip()
print("Recommendation:\n")
print(recommendation)

## Step 8: Apply Bedrock Guardrails to the response

Optional but required by the exam-question solution. If the
`OrgIQ-Guardrails` stack is deployed and notebook 05 has populated
`GUARDRAIL_ID`, the recommendation text passes through `ApplyGuardrail`
before it leaves the pipeline. Filters PII leakage, off-topic content,
and any of the denied topics configured on the guardrail.

In [ ]:
if GUARDRAIL_ID:
    gr = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=str(GUARDRAIL_VERSION),
        source="OUTPUT",
        content=[{"text": {"text": recommendation}}],
    )
    action = gr["action"]
    print(f"✓ Guardrail action: {action}")
    if action == "GUARDRAIL_INTERVENED":
        print("  (response was modified — final text below)")
        recommendation = gr["outputs"][0]["text"]
        print(recommendation)
    else:
        print("  (response passed through unmodified)")
else:
    print("⚠ GUARDRAIL_ID not set — skipping. Run notebook 05 to create one,")
    print("  then re-run this notebook to see the guardrail step in action.")

## Step 9: Step Functions + Lambda orchestration (described, not deployed)

In production, the seven steps above each become a **Lambda** wired
together by **Step Functions**. The state machine is the place where
retries, validation gates, and audit trails live — not inside any one
Lambda.

```
┌────────────────────────────────────────────────────────────────┐
│ StateMachine: CabinUpgradeRecommendation                       │
├────────────────────────────────────────────────────────────────┤
│  InterpretIntent      (Lambda → Bedrock Converse)              │
│  GenerateSQL          (Lambda → Bedrock Converse)              │
│  ValidateSQL          (Lambda — pure Python, no Bedrock)       │
│    └─ on fail: Retry GenerateSQL with stricter prompt (max 2x) │
│  ExecuteSQL           (Lambda → RDS Data API)                  │
│    └─ on fail: route to FallbackTemplatedQuery                 │
│  GenerateRecommendation (Lambda → Bedrock Converse)            │
│  ApplyGuardrail       (Lambda → Bedrock ApplyGuardrail)        │
│    └─ if INTERVENED → log + downgrade to "no upgrade"          │
│  EmitOffer            (Lambda → SQS / SNS to the app)          │
└────────────────────────────────────────────────────────────────┘
```

Why this shape:

- **Step Functions** carries the retry policy (exponential backoff on
  Bedrock throttles, bounded retries on SQL regeneration). Putting
  retries inside a Lambda would either time out or burn budget.
- **Lambdas stay single-purpose**, which means each one has a tight IAM
  policy. `ExecuteSQL` is the only Lambda with the Aurora secret;
  `ApplyGuardrail` is the only one with `bedrock:ApplyGuardrail`.
- The **validator runs after the LLM, not as a prompt instruction.**
  Prompt-level guardrails are advisory; code-level validation is
  enforceable.
- The **fallback templated query** matters: when text-to-SQL fails
  validation twice in a row, the orchestrator falls back to a
  hand-written parameterized query so the customer still gets a
  recommendation — degraded, but not silent.

## Why this beats the other options on the exam question

- **Option A (RAG over RDS)** — wrong shape. Aggregate questions over
  structured data (booking counts, average fare) can't be served from
  embedding similarity; you'd be approximating values that exist
  exactly in the database.
- **Option C (OpenSearch vectors)** — same problem as A, plus extra
  infrastructure to keep in sync with the source of truth.
- **Option D (text-to-SQL but with "confidence scoring and semantic
  similarity" instead of Step Functions)** — the orchestration tools
  named in D don't actually orchestrate retries or validation gates.
  Confidence scoring is a *signal*; it's not a workflow.

**Option B** is correct because the orchestration tools match the
*shape of the failure modes* — bounded retries, validation gates,
fallback paths — and text-to-SQL preserves the exact-row guarantee that
upgrade offers need to be defensible to airline partners.

## Step 10: Cleanup

Drops the demo tables. Safe to re-run the whole notebook; the seeds
above use `ON CONFLICT DO NOTHING`.

In [ ]:
exec_sql("DROP TABLE IF EXISTS demo_airline_bookings;")
exec_sql("DROP TABLE IF EXISTS demo_airline_customers;")
print("✓ Demo tables dropped")

## Summary

- [x] Seeded a small airline booking schema in Aurora (RDS Data API)
- [x] Bedrock interprets the customer request → tables + columns needed
- [x] Nova Pro generates parameterized SQL
- [x] Validator enforces SELECT-only, single-statement, allow-listed tables
- [x] RDS Data API executes the validated SQL
- [x] Bedrock generates a grounded recommendation from the returned rows
- [x] Bedrock Guardrails filter the output (when `GUARDRAIL_ID` is set)
- [x] Step Functions / Lambda orchestration sketched for production